# Channel mixing for LITEMV on the wide UEA datasets (the improvement run)

This notebook is preset to `MIX = 16`: the authors' LITEMV with a learned 16-channel mixing layer in front, run with the authors' own `main.py` (5 runs, 1,500 epochs) only on the UEA datasets with more than 16 channels (DuckDuckGeese, FaceDetection, Heartbeat, MotorImagery, FingerMovements, NATOPS; NATOPS' 24 channels is from memory, check it). It is the same notebook as `lite_uea_part0.ipynb`, which runs the reproduction with `MIX = 0` (the authors' unchanged LITEMV); the comparison of the two kinds of run happens in the merge cell (section 5), which reads both result files.

**What the change is.** Input (T x C) -> bias-free 1x1 convolution C -> 16 -> the authors' LITEMV built for 16 channels -> pooling -> softmax. The mixing layer adds C x 16 weights. The paper's Table 3 shows LITEMVTime at 18.00% on DuckDuckGeese (1,345 channels) against 62.00% for ConvTran and gives no explanation; measured earlier, the unchanged LITEMV has 1,464,801 trainable parameters there and the variant 45,685.

**How to run (own Kaggle account, Internet On, GPU on)**
1. Section 0: setup and the patches to `main.py` (adds `--mix`, `--epochs`, epoch printing, `params.txt`). Section 1: check `MIX = 16`.
2. Section 2: starts the workers in the background. Section 3b: live progress with run and epoch.
3. Section 4: saves `uea_part<N>_mix16_results.csv` and a logs zip. Download them before a session ends.
4. Section 5: run where the `mix16` file AND the `orig` file of the reproduction (`uea_part<N>_orig_results.csv`) are available. It reports the improvement against the paper and the effect against the reproduction.

**Caveats.** Only 5 runs x 1,500 epochs is comparable with the paper; state any reduction. The result may be negative or mixed, and that is a valid finding. The custom filters now act on mixed channels. On tiny training sets (DuckDuckGeese: 50 samples) the extra weights may overfit. The first quick test (one 50-epoch run, 50 test samples: 24% against 48% for the original) proves nothing. Do not claim a cause without a control experiment.

## 0. Setup and patches

In [ ]:
import os, sys, subprocess

REPO = "/kaggle/working/LITE"
if not os.path.isdir(REPO + "/src"):
    # clone the authors' repository (needs Internet = On)
    subprocess.run(["git", "clone", "-q", "https://github.com/MSD-IRIMAS/LITE.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aeon", "codecarbon"], check=True)

# Patches to the authors' main.py (nothing else in their code is changed):
#  1) distutils no longer exists in Python 3.13, so define strtobool locally
#  2) the code never passes n_epochs, so the model would train for only 2 epochs: add --epochs (default 1500, as in the paper), print the epochs
#  3) --mix k: optional channel mixing. LITEMV is built for k channels and a 1x1 convolution in front maps the real channels to k.
#     --mix 0 (default) leaves the authors' model untouched.
#  4) write the number of trainable parameters of every model to params.txt
s = open(REPO + "/main.py").read()
if "--mix" not in s:
    s = s.replace("from distutils.util import strtobool",
                  "def strtobool(v):\n    return str(v).lower() in ('1','true','t','yes','y')")
    s = s.replace('parser.add_argument("--runs"',
                  'parser.add_argument("--epochs", type=int, default=1500)\n    parser.add_argument("--mix", type=int, default=0)\n    parser.add_argument("--runs"')
    s = s.replace("n_channels=n_channels,\n                n_classes=len(np.unique(ytrain)),\n            )",
                  "n_channels=(args.mix or n_channels),\n                n_classes=len(np.unique(ytrain)),\n                n_epochs=args.epochs,\n                verbose=True,\n            )")
    anchor = '        if not os.path.exists(output_directory + "loss.pdf"):'
    block = '''        if args.mix and args.classifier == "LITEMV":
            import tensorflow as tf
            _inp = tf.keras.layers.Input((length_TS, n_channels))
            _mixed = tf.keras.layers.Conv1D(args.mix, 1, use_bias=False, name="channel_mix")(_inp)
            clf.model = tf.keras.models.Model(_inp, clf.model(_mixed))
            clf.model.compile(loss="categorical_crossentropy", optimizer="Adam", metrics=["accuracy"])
        with open(output_directory + "params.txt", "w") as _f:
            _f.write(str(sum(int(np.prod(w.shape)) for w in clf.model.trainable_weights)))

'''
    assert anchor in s, "anchor for the mixing block not found: main.py differs from the version this notebook was written for"
    s = s.replace(anchor, block + anchor)
    open(REPO + "/main.py", "w").write(s)
print("patches found: epochs", s.count("n_epochs=args.epochs"), "(expect 2) | mix", s.count("args.mix"), "(expect 4)")

## 1. Settings
`MIX` chooses reproduction (0) or improvement (16). The paper's Table 3 numbers for all 30 datasets are in this cell.

In [ ]:
import os, sys, json, glob, time, shutil, subprocess, textwrap
import numpy as np
import pandas as pd

ROOT, REPO = "/kaggle/working", "/kaggle/working/LITE"

# ---- what this run does ----
MIX = 16             # (preset for the IMPROVEMENT) 0 = REPRODUCTION: the authors' LITEMV, unchanged.   16 = IMPROVEMENT: LITEMV with a 16-channel mixing layer in front
                     # (the improvement only runs on datasets with more channels than MIX). Run the notebook once with 0 and once with 16.

# ---- splitting the work (optional): every person uses the same N_PARTS and a different MY_PART ----
N_PARTS = 1          # 1 = you run everything
MY_PART = 0          # 0 .. N_PARTS-1
PART_LABEL = str(MY_PART)   # used in file and folder names. Another label (for example "missing") keeps a re-run of missing datasets from clashing with the first files
ONLY = []            # empty = the normal dataset list. Otherwise run exactly these datasets (LEFT_OUT and SKIP_ORIG are then ignored)

EPOCHS, RUNS = 1500, 5                  # the paper's settings; lowering RUNS or EPOCHS must be stated in the report
N_WORKERS = 2                           # processes at the same time (one per GPU); raise to 4 only for small datasets
KIND = f"mix{MIX}" if MIX else "orig"
MODEL = f"LITEMV+mix{MIX}" if MIX else "LITEMV"
TAG = f"{KIND}_part{PART_LABEL}"           # names the output folders, so the two kinds of run and the parts never clash
WAIT, TIME_BUDGET_H = False, 11.0       # WAIT=True for "Save & Run All": keeps the notebook alive until finished
assert 0 <= MY_PART < N_PARTS, "MY_PART must be between 0 and N_PARTS-1"

# LITEMVTime accuracy (%) for all 30 UEA datasets, Table 3 of the paper
PAPER = {"FaceDetection": 61.01, "InsectWingbeat": 61.72, "PenDigits": 98.86, "SpokenArabicDigits": 98.59,
         "LSST": 66.42, "FingerMovements": 56.00, "MotorImagery": 53.00, "SelfRegulationSCP1": 73.04,
         "Heartbeat": 61.46, "SelfRegulationSCP2": 55.00, "PhonemeSpectra": 15.81, "CharacterTrajectories": 99.58,
         "EthanolConcentration": 69.20, "HandMovementDirection": 35.14, "PEMS-SF": 79.19, "RacketSports": 73.68,
         "Epilepsy": 99.28, "JapaneseVowels": 96.49, "NATOPS": 90.00, "EigenWorms": 93.89,
         "UWaveGestureLibrary": 84.68, "Libras": 89.44, "ArticularyWordRecognition": 97.33, "BasicMotions": 100.0,
         "DuckDuckGeese": 18.00, "Cricket": 98.61, "Handwriting": 40.00, "ERing": 84.44,
         "AtrialFibrillation": 13.33, "StandWalkJump": 66.67}

# left out (state this in the report). PEMS-SF: its data file did not download completely. InsectWingbeat: about 25,000 training samples.
LEFT_OUT = {"PEMS-SF", "InsectWingbeat"}
# the original LITEMV is too slow on this one (measured: about 48 minutes for 50 epochs on a T4, so about a day per run at 1500 epochs)
SKIP_ORIG = {"DuckDuckGeese"}
# datasets with MORE than 16 channels, where channel mixing can be applied (FaceDetection 144, MotorImagery 64, Heartbeat 61,
# FingerMovements 28, DuckDuckGeese 1345: seen in my shapes run. NATOPS 24: from memory, check it)
WIDE = ["DuckDuckGeese", "FaceDetection", "Heartbeat", "MotorImagery", "FingerMovements", "NATOPS"]
# datasets the authors' loader probably cannot read (unequal series lengths): tried first, they crash within seconds
# (SpokenArabicDigits and CharacterTrajectories already failed with that error in my shapes run; JapaneseVowels: from memory)
LOADER_RISK = ["SpokenArabicDigits", "CharacterTrajectories", "JapaneseVowels"]
# large datasets or long series (my judgment): last
HEAVY = ["PenDigits", "FaceDetection", "LSST", "PhonemeSpectra", "MotorImagery", "EigenWorms", "Heartbeat",
         "SelfRegulationSCP1", "SelfRegulationSCP2", "EthanolConcentration"]

# names the merge cell needs
MODE, MIX_DIMS, EPOCHS_FOR = "full", [MIX or 16], {}
print(f"{MODEL} | part {MY_PART} of {N_PARTS} | {RUNS} runs x {EPOCHS} epochs | workers {N_WORKERS}")

## 2. Check the split and start the runs in the background
Builds the dataset list of this run, checks that the parts do not overlap, prints your datasets and starts a shared queue: each worker takes the next free dataset. Rerunning this cell resumes: finished datasets are skipped and crashed ones are retried.

In [ ]:
if ONLY:
    names = list(ONLY)                                                                 # explicit list: exactly these datasets
elif MIX:
    names = [d for d in WIDE if d not in LEFT_OUT]                                    # improvement: only the wide datasets
else:
    names = [d for d in PAPER if d not in LEFT_OUT and d not in SKIP_ORIG]           # reproduction: all that are practical
ordered = ([d for d in names if d in LOADER_RISK]
           + [d for d in names if d not in LOADER_RISK and d not in HEAVY]
           + [d for d in names if d in HEAVY])
queue = ordered[MY_PART::N_PARTS]          # alternating split: every part gets a similar mix of cheap and costly datasets

# split check: the parts must not overlap and must together cover every dataset of this run
parts_all = [ordered[p::N_PARTS] for p in range(N_PARTS)]
assert sum(len(p) for p in parts_all) == len(set().union(*map(set, parts_all))) == len(names), "split check failed"
print(f"{MODEL}: {len(names)} datasets | parts: {N_PARTS} | part sizes: {[len(p) for p in parts_all]}")
print(f"my share (part {MY_PART}): {len(queue)} datasets:", queue)

CLAIMS, LOGS = f"{ROOT}/claims_{TAG}/", f"{ROOT}/logs_{TAG}/"
os.makedirs(CLAIMS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)

# resume: datasets with a finished result are skipped. Results of an EARLIER session can be added as a Kaggle dataset input
# (the file written by the save cell) and are skipped too.
done = set()
for f in glob.glob(f"{ROOT}/uea_{TAG}_w*/LITEMV/results_ucr.csv"):
    done |= set(pd.read_csv(f)["dataset"])
for f in (glob.glob(f"/kaggle/input/**/uea_part{PART_LABEL}_{KIND}_results.csv", recursive=True) + glob.glob(f"/kaggle/input/**/uea_part{PART_LABEL}_{KIND}_results.csv", recursive=True)):
    done |= set(pd.read_csv(f)["dataset"])
for c in os.listdir(CLAIMS):               # a claim without a result means that run crashed: release it
    if c not in done:
        os.remove(CLAIMS + c)
queue = [d for d in queue if d not in done]
json.dump(queue, open(f"{ROOT}/queue_{TAG}.json", "w"))
print(f"already finished: {len(done)} | to run now: {len(queue)}")

# worker script: claims the next free dataset (atomic file creation), runs the authors' main.py on it, repeats
open(f"{ROOT}/worker_{TAG}.py", "w").write(textwrap.dedent("""
    import os, sys, json, subprocess
    w, root, repo, tag, epochs, runs, mix = int(sys.argv[1]), sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5], sys.argv[6], sys.argv[7]
    out = f"{root}/uea_{tag}_w{w}/"; os.makedirs(out, exist_ok=True)       # one output folder per worker
    claims, logs = f"{root}/claims_{tag}/", f"{root}/logs_{tag}/"
    for d in json.load(open(f"{root}/queue_{tag}.json")):
        try:
            os.close(os.open(claims + d, os.O_CREAT | os.O_EXCL))        # fails if another worker already took it
        except FileExistsError:
            continue
        cmd = [sys.executable, "main.py", "--dataset", d, "--classifier", "LITEMV", "--runs", runs, "--epochs", epochs,
               "--output-directory", out] + (["--mix", mix] if mix != "0" else [])
        with open(logs + d + ".log", "w") as lg:
            subprocess.run(cmd, cwd=repo, stdout=lg, stderr=subprocess.STDOUT)
"""))

procs = []
for w in range(N_WORKERS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(w % 2), TF_FORCE_GPU_ALLOW_GROWTH="true", TF_CPP_MIN_LOG_LEVEL="3")
    procs.append(subprocess.Popen([sys.executable, f"{ROOT}/worker_{TAG}.py", str(w), ROOT, REPO, TAG, str(EPOCHS), str(RUNS), str(MIX)],
                                  env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                                  start_new_session=True))       # own process group: pressing stop on a notebook cell no longer kills the workers
print("started", N_WORKERS, "workers")

if WAIT:                                   # only for Commit runs: block until finished or the time budget is used
    t0 = time.time()
    while any(p.poll() is None for p in procs) and (time.time() - t0) / 3600 < TIME_BUDGET_H:
        time.sleep(60)

## 3. Progress and time left (short text summary)

In [ ]:
parts = [pd.read_csv(f) for f in glob.glob(f"{ROOT}/uea_{TAG}_w*/LITEMV/results_ucr.csv")]
res = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["dataset", "Training Time-mean"])
queue = json.load(open(f"{ROOT}/queue_{TAG}.json"))
done = set(res["dataset"])
todo = [d for d in queue if d not in done]
claimed = set(os.listdir(CLAIMS))
alive = max(int(subprocess.run(f"pgrep -fc worker_{TAG}.py", shell=True, capture_output=True, text=True).stdout.strip() or 0) - 1, 0)  # minus the check command itself

print("workers alive:", alive, "| finished this session:", len(done), "| still to do:", len(todo))
if len(res):
    per_dataset = float(res["Training Time-mean"].mean()) * RUNS       # seconds per dataset, measured under the same parallel load
    print(f"average {per_dataset/60:.1f} min per dataset -> about {len(todo) * per_dataset / N_WORKERS / 3600:.1f} hours left (rough, optimistic early on)")
print("running now (claimed, no result yet):", sorted(claimed - done))

## 3b. Live progress (real time)
Overall progress bar, one bar per running dataset (run and epoch), datasets finished, time left, a CRASHED list with the error, and a warning if the workers have died. It runs for 20 minutes and then stops by itself (run it again to keep watching); the runs are not affected. It blocks the notebook while it runs, so do not press stop on it.

In [ ]:
# ---------- live progress for the UCR run: bars per dataset, overall progress, time left, stalled-job warning ----------
import re, glob, json, os, time, subprocess, collections
import pandas as pd
from IPython.display import HTML, display, clear_output

_seen = {}     # epochs seen per dataset, so progress never goes backwards
_hist = []     # (time, units done) readings, used for the speed and the time-left estimate

def _bar(frac, width=28):
    n = int(round(width * max(0.0, min(1.0, frac))))
    return "█" * n + "░" * (width - n)

def snapshot():
    queue = json.load(open(f"{ROOT}/queue_{TAG}.json"))                        # datasets to run in this session
    finished = set()
    for f in glob.glob(f"{ROOT}/uea_{TAG}_w*/LITEMV/results_ucr.csv"):
        finished |= set(pd.read_csv(f)["dataset"])                             # a dataset is finished when its results row exists
    runs_done = collections.Counter(p.split("/")[-2] for p in
                                    glob.glob(f"{ROOT}/uea_{TAG}_w*/LITEMV/run_*/*/dict_emissions.json"))   # one file per finished run
    running, crashed = [], []
    for d in sorted(set(os.listdir(CLAIMS)) - finished):                       # claimed by a worker, not finished
        log, age, n_ep = f"{LOGS}{d}.log", None, 0
        if os.path.exists(log):
            age = (time.time() - os.path.getmtime(log)) / 60                    # minutes since the log last changed
            with open(log, "rb") as fh:
                data = fh.read()                                               # the WHOLE log: warning lines can push the epoch lines out of any short tail
            n_ep = len(re.findall(rb"Epoch \d+/\d+", data))                    # every epoch of every run prints one "Epoch n/E" line
            txt = data[-6000:].decode("utf-8", "ignore")
            if "Traceback (most recent call last)" in txt and age > 2:           # the process ended with an error
                last = [l.strip() for l in txt.splitlines() if l.strip()]
                err = next((l for l in reversed(last) if "Error" in l or "Exception" in l), last[-1])
                crashed.append(dict(dataset=d, error=err[:110]))
                continue
        n_ep = max(n_ep, _seen.get(d, 0)); _seen[d] = n_ep                      # progress never goes backwards
        if n_ep > 0 and n_ep % EPOCHS == 0:
            run, ep = min(n_ep // EPOCHS, RUNS), EPOCHS                          # a run has just finished
        else:
            run, ep = min(n_ep // EPOCHS + 1, RUNS), n_ep % EPOCHS
        running.append(dict(dataset=d, run=run, epoch=ep, units=n_ep, log_age_min=age))
    per_ds = RUNS * EPOCHS
    done_units = len([d for d in queue if d in finished]) * per_ds + sum(min(r["units"], per_ds) for r in running)
    alive = max(int(subprocess.run(f"pgrep -fc worker_{TAG}.py", shell=True, capture_output=True, text=True).stdout.strip() or 0) - 1, 0)
    return dict(queue=queue, finished=finished, running=running, crashed=crashed, done_units=done_units,
                total_units=(len(queue) - len(crashed)) * per_ds, alive=alive)     # crashed datasets will not finish: not counted

def render(s):
    now = time.time()
    _hist.append((now, s["done_units"]))
    while _hist and now - _hist[0][0] > 900:                                   # speed from the last 15 minutes
        _hist.pop(0)
    rate = (s["done_units"] - _hist[0][1]) / ((now - _hist[0][0]) / 60) if now - _hist[0][0] > 90 else None   # epochs per minute, all workers together
    left = s["total_units"] - s["done_units"]
    eta = f"about {left / rate / 60:.1f} h left (rough: big datasets take longer per epoch)" if rate and rate > 0 else "time left: measuring ..."
    frac = s["done_units"] / s["total_units"] if s["total_units"] else 1.0
    n_fin = len([d for d in s["queue"] if d in s["finished"]])
    lines = [f"{time.strftime('%H:%M:%S')} | part {MY_PART} | workers alive: {s['alive']} | {RUNS} runs x {EPOCHS} epochs",
             f"OVERALL  [{_bar(frac)}] {100 * frac:5.1f}% | datasets finished {n_fin}/{len(s['queue'])} | crashed {len(s['crashed'])} | {eta}", "",
             "RUNNING NOW"]
    for r in s["running"]:
        stalled = r["log_age_min"] is not None and r["log_age_min"] > 10 and s["alive"] > 0
        f2 = min(r["units"] / (RUNS * EPOCHS), 1.0)
        lines.append(f"  {r['dataset']:<26s}[{_bar(f2, 20)}] run {r['run']}/{RUNS} epoch {r['epoch']:>4d}/{EPOCHS}"
                     + ("   <-- NO LOG ACTIVITY for %.0f min: stalled or crashed?" % r["log_age_min"] if stalled else ""))
    if not s["running"]:
        lines.append("  (nothing running)")
    if s["crashed"]:
        lines += ["", f"CRASHED ({len(s['crashed'])}): list these as not reproduced, with the error"]
        lines += [f"  {c['dataset']:<26s}{c['error']}" for c in s["crashed"]]
    lines += ["", "FINISHED: " + (", ".join(sorted(d for d in s["queue"] if d in s["finished"])) or "none yet")]
    if s["alive"] == 0 and s["running"]:
        lines.append("\nWARNING: no workers alive but some datasets are unfinished: they crashed. Rerun the launcher cell to retry them.")
    return "<pre style='font-family:monospace;font-size:12px;line-height:1.35'>" + "\n".join(lines) + "</pre>"

def live(minutes=20, every=15):
    # shows a live-updating display for `minutes` minutes (or until everything is finished); blocks the notebook while it runs
    t0 = time.time()
    while True:
        s = snapshot()
        clear_output(wait=True)
        display(HTML(render(s)))
        if s["total_units"] and s["done_units"] >= s["total_units"]:
            print("ALL FINISHED: run the save and compare cells"); break
        if time.time() - t0 > minutes * 60:
            print(f"(stopped after {minutes} min: run live() again to keep watching; the runs are not affected)"); break
        time.sleep(every)

live(minutes=20, every=15)

## 4. Save my part
Writes `uea_part<MY_PART>_<orig|mix16>_results.csv` (percent accuracies, one row per dataset, this session plus earlier sessions) and a zip of the logs. Download both from the Output panel.

In [ ]:
# Save MY part in one tidy file (percent accuracies, one row per dataset) that the merge cell and compare_uea.py read.
# The file of an earlier session (uploaded as a Kaggle input) is added, and a newer result for a dataset replaces an older one.
frames = []
for f in glob.glob(f"{ROOT}/uea_{TAG}_w*/LITEMV/results_ucr.csv"):
    d = pd.read_csv(f)
    wdir = os.path.dirname(os.path.dirname(f))
    params = []
    for ds in d["dataset"]:
        p = f"{os.path.dirname(f)}/run_0/{ds}/params.txt"
        params.append(int(open(p).read()) if os.path.exists(p) else np.nan)
    frames.append(pd.DataFrame(dict(dataset=d["dataset"], model=MODEL, runs=RUNS, epochs=EPOCHS,
                                    single_mean=100 * d["LITEMV-mean"], single_std=100 * d["LITEMV-std"],
                                    ensemble=100 * d["LITEMVTime"], train_sec=d["Training Time-mean"], params=params)))
for f in (glob.glob(f"/kaggle/input/**/uea_part{PART_LABEL}_{KIND}_results.csv", recursive=True) + glob.glob(f"/kaggle/input/**/uea_part{PART_LABEL}_{KIND}_results.csv", recursive=True)):
    frames.append(pd.read_csv(f))
if frames:
    mine = pd.concat(frames, ignore_index=True).drop_duplicates("dataset", keep="first")
    mine.to_csv(f"{ROOT}/uea_part{PART_LABEL}_{KIND}_results.csv", index=False)
    shutil.make_archive(f"{ROOT}/uea_part{PART_LABEL}_{KIND}_logs", "zip", LOGS)
    print(len(mine), "datasets saved to", f"{ROOT}/uea_part{PART_LABEL}_{KIND}_results.csv")
    print("download that file and the logs zip from the Output panel (the logs hold the error of every dataset that crashed)")
else:
    print("no results yet")

## 5. Merge, compare with the paper, Excel file
Reads every `uea_part*_results.csv` it finds (also as Kaggle inputs), so run it where the `orig` and `mix16` files of all parts are available. Prints reproduction, improvement and effect, compares the authors' CSV with Table 3, and writes `uea_paper_vs_my_results.xlsx` and `uea_comparison_with_paper.csv`.

In [ ]:
from scipy.stats import wilcoxon
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ---- collect every result: part files (from all people, also as Kaggle inputs) and this session's raw rows ----
files = (glob.glob(f"{ROOT}/uea_part*_results.csv") + glob.glob("/kaggle/input/**/uea_part*_results.csv", recursive=True)
         + glob.glob("/kaggle/input/**/uea_part*_results.csv", recursive=True))
rows = []
for f in files:
    rows += pd.read_csv(f).to_dict("records")
for f in glob.glob(f"{ROOT}/uea_results_*_w*.jsonl"):
    rows += [json.loads(l) for l in open(f) if l.strip()]
allr = pd.DataFrame(rows).drop_duplicates(["dataset", "model"], keep="last")
print("result files used:", files, "| rows:", len(allr))

if "error" in allr.columns and allr["error"].apply(lambda e: isinstance(e, str)).any():
    bad = allr[allr["error"].apply(lambda e: isinstance(e, str))]
    print("\nFailed jobs (report them):"); print(bad[["dataset", "model", "error"]].to_string(index=False))
    ok = allr.drop(bad.index)
else:
    ok = allr

D = list(PAPER)                                      # all 30 datasets of Table 3
def pick(model, col):
    s = ok[ok["model"] == model].drop_duplicates("dataset", keep="last").set_index("dataset")[col]
    return s.reindex(D)                              # NaN where this model was not run

mix_model = f"LITEMV+mix{MIX_DIMS[0]}"
tbl = pd.DataFrame(index=D)
ch = ok.dropna(subset=["channels"]).drop_duplicates("dataset").set_index("dataset")["channels"] if "channels" in ok.columns else pd.Series(dtype=float)
tbl["channels"] = ch.reindex(D)
tbl["paper"] = pd.Series(PAPER)
tbl["orig_ens"] = pick("LITEMV", "ensemble")
tbl["orig_single"] = pick("LITEMV", "single_mean")
tbl["orig_std"] = pick("LITEMV", "single_std")
tbl["mix_ens"] = pick(mix_model, "ensemble")
tbl["mix_single"] = pick(mix_model, "single_mean")
tbl["mix_std"] = pick(mix_model, "single_std")
tbl["orig_minus_paper"] = tbl["orig_ens"] - tbl["paper"]
tbl["mix_minus_paper"] = tbl["mix_ens"] - tbl["paper"]
tbl["mix_minus_orig"] = tbl["mix_ens"] - tbl["orig_ens"]
tbl["params_orig"] = pick("LITEMV", "params")
tbl["params_mix"] = pick(mix_model, "params")
print(tbl.round(2).to_string())
tbl.to_csv(f"{ROOT}/uea_comparison_with_paper.csv")

def summarize(label, d):
    d = d.dropna()
    if len(d):
        print(f"{label}: {len(d)} datasets | mean difference {d.mean():+.2f} points | wins/ties/losses "
              f"{int((d>0).sum())}/{int((d==0).sum())}/{int((d<0).sum())}" +
              (f" | Wilcoxon p = {wilcoxon(d[d != 0]).pvalue:.4f}" if (d != 0).sum() >= 6 else ""))
print()
summarize("REPRODUCTION  (my rerun of the authors' LITEMV minus the paper)", tbl["orig_minus_paper"])
summarize("IMPROVEMENT   (channel mixing minus the paper)", tbl["mix_minus_paper"])
summarize("EFFECT        (channel mixing minus my rerun of the original)", tbl["mix_minus_orig"])
print("not run (left out, skipped or failed):", sorted(tbl[tbl["orig_ens"].isna() & tbl["mix_ens"].isna()].index))

# ---- the authors' own CSV, compared with Table 3 ----
try:
    auth = pd.read_csv(REPO + "/results_multivariate.csv")
    cand = [c for c in auth.columns if "litemv" in c.lower() and "time" in c.lower()]
    print("\nauthors' CSV columns:", list(auth.columns), "| detected LITEMVTime column:", cand)
    if cand:
        a = auth.set_index(auth.columns[0])[cand[0]]
        if a.max() <= 1.0: a = a * 100               # stored as fractions -> percent
        cmp_ = pd.DataFrame({"table3_paper": pd.Series(PAPER), "authors_csv": a.reindex(D)})
        cmp_["csv_minus_table3"] = cmp_["authors_csv"] - cmp_["table3_paper"]
        print(cmp_.round(2).to_string())
except Exception as e:
    print("authors' CSV not compared:", type(e).__name__, e)

# ---- Excel file: the paper's results next to my results (differences are formulas) ----
F = "Arial"
base, bold, blue = Font(name=F, size=10), Font(name=F, size=10, bold=True), Font(name=F, size=10, color="0000FF")
hdr_font, hdr_fill = Font(name=F, size=10, bold=True, color="FFFFFF"), PatternFill("solid", start_color="305496")
paper_fill, mine_fill = PatternFill("solid", start_color="FCE4D6"), PatternFill("solid", start_color="E2EFDA")
thin = Side(style="thin", color="BFBFBF"); box = Border(left=thin, right=thin, top=thin, bottom=thin)
wb = Workbook(); ws = wb.active; ws.title = "Paper vs my results"
ws["A1"] = "LITEMV on all UEA datasets: the paper's results vs my results"; ws["A1"].font = Font(name=F, size=14, bold=True)
ws["A2"] = (f"Accuracy in %. Orange = LITEMVTime, Table 3 of the paper. Green = my runs (mode '{MODE}': {RUNS} runs, {EPOCHS} epochs). "
            "Only full-mode numbers (5 runs, 1500 epochs) are comparable with the paper. Blank = not run.")
ws["A2"].font = base
heads = ["Dataset", "Channels", "Paper (Table 3)", "Original LITEMV (my rerun)", f"Improvement ({mix_model})",
         "Original - paper", "Improvement - paper", "Improvement - my rerun", "Params original", "Params improvement"]
for j, h in enumerate(heads, 1):
    c = ws.cell(4, j, h); c.font, c.fill, c.border = hdr_font, hdr_fill, box
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
ws.row_dimensions[4].height = 45
num = lambda v: None if pd.isna(v) else float(v)
for i, d in enumerate(D):
    r = 5 + i
    for j, v in enumerate([d, num(tbl.loc[d, "channels"]), PAPER[d], num(tbl.loc[d, "orig_ens"]), num(tbl.loc[d, "mix_ens"])], 1):
        ws.cell(r, j, v).font = bold if j == 1 else blue
    ws.cell(r, 6, f'=IF(COUNT(D{r},C{r})=2,D{r}-C{r},"")')
    ws.cell(r, 7, f'=IF(COUNT(E{r},C{r})=2,E{r}-C{r},"")')
    ws.cell(r, 8, f'=IF(COUNT(E{r},D{r})=2,E{r}-D{r},"")')
    ws.cell(r, 9, num(tbl.loc[d, "params_orig"])).font = blue
    ws.cell(r, 10, num(tbl.loc[d, "params_mix"])).font = blue
    for j in range(1, 11):
        c = ws.cell(r, j); c.border = box
        if c.font.name != F: c.font = base
        c.alignment = Alignment(horizontal="left" if j == 1 else "center")
    ws.cell(r, 3).fill = paper_fill
    for j in (4, 5): ws.cell(r, j).fill = mine_fill
    for j in (3, 4, 5): ws.cell(r, j).number_format = "0.00"
    for j in (6, 7, 8): ws.cell(r, j).number_format = "+0.00;-0.00;0.00"
    for j in (2, 9, 10): ws.cell(r, j).number_format = "#,##0"
n0 = 5 + len(D) + 1
for k, t in enumerate(["Notes",
        "Blue = value copied from my runs or the paper. Black = formula.",
        "Original - paper is the REPRODUCTION. Improvement - paper is the comparison asked for. Improvement - my rerun isolates the effect of the change.",
        "Paper = LITEMVTime (5-model ensemble). Mine = ensemble of my runs. The improvement only exists on datasets with more channels than the mixing size.",
        "Differences within the run-to-run spread (single_std in the CSV) should not be called wins."]):
    ws.cell(n0 + k, 1, t).font = bold if k == 0 else base
for j, w in enumerate([26, 10, 14, 18, 20, 14, 18, 20, 15, 17], 1):
    ws.column_dimensions[get_column_letter(j)].width = w
ws.freeze_panes = "B5"
wb.save(f"{ROOT}/uea_paper_vs_my_results.xlsx")
print("\nsaved:", f"{ROOT}/uea_paper_vs_my_results.xlsx", "and", f"{ROOT}/uea_comparison_with_paper.csv")